# Fake News Detection — Pipeline Machine Learning & Ekspor Android

Repository: **Central-Saga/Deny-Project3-ML-FakeNewsDetection**  
Modul: **Machine Learning Pipeline (src/) & Reproducibility Audit (Issue #13)**

---

## 1. Konteks & Latar Belakang

Aplikasi Android **FakeNews AI** memuat dua pasang artefak model offline:
1. `fake_news_model_en_v13.onnx` (587 KB) + `fake_news_model_en_v13.json` (8,2 MB) — Model Bahasa Inggris LinearSVC Dual TF-IDF (100k word + 50k char).
2. `fake_news_model_id_political_v3.onnx` (391 KB) + `fake_news_model_id_political_v3.json` (4,4 MB) — Model Bahasa Indonesia Political LinearSVC (100k word).

Sesuai **Standar Project Production-Ready**, seluruh artefak produksi yang dikirim ke pengguna akhir harus dapat dihasilkan ulang (*fully reproducible*) dari *clean clone* melalui kode modular yang terpisah (`src/`), didukung dependensi terpin (`requirements-src.txt`), memiliki evaluasi langsung pada artefak ONNX, serta diverifikasi dengan **uji paritas numerik Python vs Kotlin** (toleransi $\le 10^{-3}$) untuk mengunci perilaku preprocessing 1.055 baris di `FakeNewsClassifier.kt`.

### Alur Pipeline Produksi:

```
[Dataset Mentah di data/] 
       │
       ▼
 [src/train.py]  ─────────────► [models/*.pkl] (LinearSVC + Vectorizer)
                                      │
                                      ▼
                             [src/export_android.py]
                                      │
                     ┌────────────────┴────────────────┐
                     ▼                                 ▼
         [*.onnx (Model Graph)]             [*.json (Vocab, IDF, Coef)]
                     │                                 │
                     └────────────────┬────────────────┘
                                      │
                                      ▼
                        [android-app/app/src/main/assets/]
                                      │
                     ┌────────────────┴────────────────┐
                     ▼                                 ▼
      [Inference Aplikasi Android]           [src/evaluate.py]
        (FakeNewsClassifier.kt)                        │
                     │                                 ▼
                     ▼                        [metrics.json & Uji Paritas]
        [Uji Paritas Kotlin Test] <──────────> (Toleransi 1e-3 vs Fixture)
```


## 2. Setup Lingkungan & Dependensi Terpin

Seluruh dependensi dipin pada `requirements-src.txt` guna menjamin determinisme antar platform.


In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import sklearn
import onnx
import onnxruntime as ort

print(f"Python version      : {sys.version.split()[0]}")
print(f"NumPy version       : {np.__version__}")
print(f"Pandas version      : {pd.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"ONNX version        : {onnx.__version__}")
print(f"ONNX Runtime version: {ort.__version__}")


## 3. Preprocessing Teks & Paritas 1-to-1 dengan Kotlin (`FakeNewsClassifier.kt`)

Salah satu tantangan kritis ML di Android adalah *training-serving skew* pada tahap pembersihan teks. Logika preprocessing Python di `src/evaluate.py` diselaraskan secara presisi dengan implementasi Kotlin:

### English Model v1.3:
1. **Lowercase**: Menyeragamkan seluruh huruf ke huruf kecil.
2. **URL Removal**: Menghapus `http\S+|www\S+`.
3. **HTML Tag Removal**: Menghapus tag `<.*?>`.
4. **Reuters Tag Removal**: Menghapus pola `\breuters\b` (mencegah kebocoran data sumber).
5. **Number Stripping**: Menghapus digit angka `\d+`.
6. **Punctuation Deletion**: Karakter pungtuasi dihapus (bukan diganti spasi).
7. **Word TF-IDF**: Token `[\w]{2,}`, penyaringan stop words, dan ekstraksi unigram + bigram.
8. **Char TF-IDF**: Ekstraksi `char_wb` n-gram 3..5 pada kata ber-padding (`" $word "`).
9. **Sublinear TF**: $TF = 1.0 + \ln(\text{count})$.
10. **L2 Normalization**: Vektor dinormalisasi $\frac{v}{\|v\|_2}$.

### Indonesian Political v3:
1. Pembersihan URL, email (`\b[\w\.-]+@[\w\.-]+\.\w+\b`), dan nama domain.
2. Penghapusan pola boilerplate cek fakta (seperti `turnbackhoax`, `cek fakta`, `hasil penelusuran`).
3. Penghapusan karakter non-alfanumerik.
4. Tokenisasi kata `[a-z0-9]{2,}` unigram + bigram dan normalisasi L2.


In [ ]:
from src.evaluate import (
    clean_english_text,
    create_english_word_counts,
    create_english_char_counts,
    clean_indonesian_text,
    create_indonesian_word_counts,
    create_tfidf_vector,
)

sample_en = "WASHINGTON (Reuters) - 100 new economic bills passed! Check https://reuters.com/deal."
cleaned_en = clean_english_text(sample_en)
print("=== DEMO PREPROCESSING ENGLISH ===")
print("Input  :", sample_en)
print("Cleaned:", cleaned_en)

sample_id = "Simak hasil penelusuran cek fakta di https://turnbackhoax.id terkait klaim bansos 100 juta rupiah!"
cleaned_id = clean_indonesian_text(sample_id)
print("\n=== DEMO PREPROCESSING INDONESIA ===")
print("Input  :", sample_id)
print("Cleaned:", cleaned_id)


## 4. Pelatihan Model via `src/train.py`

Logika pelatihan diekstrak ke `src/train.py` dengan parameter CLI `--data-dir`, `--out-dir`, dan `--seed`.
Model menggunakan `LinearSVC(C=1.0)` yang optimal untuk klasifikasi teks berdimensi tinggi:
- **English**: Input gabungan `hstack([X_train_word, X_train_char])` berdimensi $100.000 + 50.000 = 150.000$ fitur.
- **Indonesian**: TF-IDF unigram+bigram berdimensi $100.000$ fitur.

Perintah CLI:
```bash
python src/train.py --data-dir data/ --out-dir models/
```


In [ ]:
import subprocess

print("Memeriksa kesiapan script src/train.py:")
result = subprocess.run([sys.executable, "src/train.py", "--help"], capture_output=True, text=True)
print(result.stdout[:350] + "...")


## 5. Ekspor ke Format ONNX + JSON via `src/export_android.py`

Aplikasi Android tidak menjalankan Python, melainkan menggunakan:
1. **JSON**: Menyimpan pemetaan kosakata (*vocabulary* $\rightarrow$ index), nilai IDF, intercept SVM, dan threshold margin.
2. **ONNX Graph**: Menyimpan komputasi bobot Linear SVM (`score = coef @ vector + intercept`) yang dieksekusi dengan akselerasi perangkat keras via `ai.onnxruntime:onnxruntime-android`.

Perintah CLI:
```bash
python src/export_android.py --model-dir models/ --out-dir android-app/app/src/main/assets/
```


In [ ]:
assets_dir = "android-app/app/src/main/assets"
artifacts = [
    "fake_news_model_en_v13.onnx",
    "fake_news_model_en_v13.json",
    "fake_news_model_id_political_v3.onnx",
    "fake_news_model_id_political_v3.json",
]

print("Pemeriksaan Artefak Model di Android Assets:")
for art in artifacts:
    p = os.path.join(assets_dir, art)
    if os.path.exists(p):
        size_kb = os.path.getsize(p) / 1024
        print(f"  [OK] {art:<36} ({size_kb:>8.1f} KB)")
    else:
        print(f"  [MISSING] {art}")


## 6. Evaluasi Artefak ONNX & Penulisan `metrics.json`

Evaluasi produksi dijalankan langsung terhadap artefak ONNX aktif (bukan file `.pkl` lama), menghasilkan `metrics.json` yang mencakup metrik performa serta matriks kebingungan (*confusion matrix*).

Perintah CLI:
```bash
python src/evaluate.py --data-dir data/ --assets-dir android-app/app/src/main/assets/
```


In [ ]:
metrics_file = os.path.join(assets_dir, "metrics.json")
with open(metrics_file, encoding="utf-8") as f:
    metrics_data = json.load(f)

print("=== HASIL EVALUASI MODEL PRODUKSI ===")
en_m = metrics_data["english_v13"]
print(f"Model English v1.3:")
print(f"  Akurasi (Confident) : {en_m['accuracy_confident'] * 100:.2f}%")
print(f"  F1-Score (Confident): {en_m['f1_confident'] * 100:.2f}%")
print(f"  Coverage            : {en_m['coverage'] * 100:.2f}%")
print(f"  Confusion Matrix    : {en_m['confusion_matrix']}")

id_m = metrics_data["indonesian_political_v3"]
print(f"\nModel Indonesian Political v3:")
print(f"  Akurasi             : {id_m['accuracy'] * 100:.2f}%")
print(f"  F1-Score            : {id_m['f1'] * 100:.2f}%")
print(f"  Confusion Matrix    : {id_m['confusion_matrix']}")


## 7. Uji Paritas Numerik Python vs Kotlin (FakeNewsClassifier.kt)

Untuk memastikan tidak ada perbedaan perilaku antara implementasi riset (Python) dan implementasi aplikasi mobile (Kotlin), dilakukan uji paritas pada sampel kalimat dengan kriteria:
$$\Delta = |\text{score}_{\text{Python}} - \text{score}_{\text{Kotlin}}| < 10^{-3}$$

Skor Kotlin diperoleh langsung dari eksekusi unit test `FakeNewsClassifierTest.kt` yang menguji 1.055 baris `FakeNewsClassifier.kt` dan disimpan ke dalam fixture ter-commit `src/parity_fixtures.json`.


In [ ]:
with open("src/parity_fixtures.json", encoding="utf-8") as f:
    fixtures = json.load(f)

en_parity = metrics_data["parity_test_en_python_vs_kotlin"]
id_parity = metrics_data["parity_test_id_python_vs_kotlin"]

print(f"Status Uji Paritas English    : {'ALL PASS' if en_parity['all_pass'] else 'FAILED'} ({en_parity['passed_count']}/{en_parity['sample_count']})")
print(f"Status Uji Paritas Indonesian : {'ALL PASS' if id_parity['all_pass'] else 'FAILED'} ({id_parity['passed_count']}/{id_parity['sample_count']})")

df_parity_en = pd.DataFrame(en_parity["samples"])
print("\nTabel Perbandingan Skor English (20 Sampel):")
try:
    display(df_parity_en[["index", "sentence", "score_python", "score_kotlin", "diff", "pass"]])
except NameError:
    print(df_parity_en[["index", "sentence", "score_python", "score_kotlin", "diff", "pass"]].to_string(index=False))


## 8. Simulasi Prediksi Berita Interaktif

Fungsi berikut mensimulasikan alur prediksi persis seperti yang dieksekusi oleh `FakeNewsClassifier.kt` pada smartphone Android.


In [ ]:
from src.evaluate import score_en_onnx

with open(os.path.join(assets_dir, "fake_news_model_en_v13.json"), encoding="utf-8") as f:
    cfg_en = json.load(f)

session_en = ort.InferenceSession(os.path.join(assets_dir, "fake_news_model_en_v13.onnx"))

def predict_english_news(headline: str, body: str = "") -> dict:
    full_text = f"{headline} {body}".strip()
    cleaned = clean_english_text(full_text)
    
    word_stop = set(cfg_en["word"].get("stop_words", []))
    wv = create_tfidf_vector(create_english_word_counts(cleaned, word_stop), cfg_en["word"]["vocabulary"], cfg_en["word"]["idf"])
    cv = create_tfidf_vector(create_english_char_counts(cleaned), cfg_en["char"]["vocabulary"], cfg_en["char"]["idf"])
    
    score = score_en_onnx(session_en, wv, cv)
    threshold = cfg_en.get("uncertain_threshold", 0.20)
    
    if score <= -threshold:
        verdict = "FAKE NEWS (0)"
    elif score >= threshold:
        verdict = "REAL NEWS (1)"
    else:
        verdict = "UNCERTAIN (-1)"
        
    return {
        "text": full_text[:60] + "...",
        "score": round(score, 4),
        "threshold": threshold,
        "verdict": verdict
    }

test_samples = [
    "NASA successfully launches new weather satellite into orbit",
    "Secret alien base confirmed on the moon by military whistleblower",
    "Local school district holds annual science fair competition"
]

for sample in test_samples:
    res = predict_english_news(sample)
    print(f"Input   : {sample}")
    print(f"Score   : {res['score']:+0.4f}  |  Hasil: {res['verdict']}\n")


## 9. Ringkasan & Status Penutupan Issue #13

Berdasarkan seluruh hasil audit dan verifikasi:
1. **Pipeline Modular**: Logika pelatihan, ekspor, dan evaluasi kini berada di `src/` (`train.py`, `export_android.py`, `evaluate.py`).
2. **Dependensi Terpin**: `requirements-src.txt` memin seluruh versi pustaka ML untuk keterulangan penuh.
3. **Uji Paritas Python vs Kotlin**:
   - `FakeNewsClassifierTest.kt` (16 unit tests + parity runner) berhasil dijalankan melalui `./gradlew test` di JVM lokal dan mengunci seluruh preprocessing teks.
   - Fixture skor Kotlin disimpan di `src/parity_fixtures.json` dan `data/parity_fixtures.json`.
   - `src/evaluate.py` memvalidasi paritas skor Python vs Kotlin pada 20 kalimat English dan 10 kalimat Indonesian dengan selisih rata-rata $\approx 10^{-8}$ (jauh di bawah batas toleransi $10^{-3}$).
4. **Notebook sebagai Narasi**: `fake_news_detection.ipynb` kini menjadi dokumen narasi komprehensif yang memanggil dan memvalidasi pipeline `src/`.
